In [7]:
from google.colab import userdata
import os
import pandas as pd

os.environ['LLM_API_KEY'] = userdata.get('LLM_API_KEY')
print("API key loaded:", "Yes" if os.environ.get('LLM_API_KEY') else "No")

API key loaded: Yes


In [8]:
import requests
import json

def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    api_key = os.environ['LLM_API_KEY']
    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "openrouter/free",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    response = requests.post(url, headers=headers, json=payload)

    if response.status_code != 200:
        print("Error:", response.status_code)
        print(response.text)
        return None

    return response.json()['choices'][0]['message']['content']

In [9]:
test_response = call_llm(
    system_prompt="You are a helpful assistant.",
    user_prompt="Reply with only the word: hello"
)
print("Test response:", test_response)

Test response: 

hello



In [10]:
# task-2
import re

def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))

# Test 1: contains an email — should be blocked
test_with_pii = "Contact me at samiksha@example.com for details about this diamond."
print("Test 1 (has email) — PII detected:", has_pii(test_with_pii))

# Test 2: clean input — should proceed
test_clean = "This diamond has carat 1.5, cut Ideal, color D, clarity VVS2."
print("Test 2 (clean) — PII detected:", has_pii(test_clean))

Test 1 (has email) — PII detected: True
Test 2 (clean) — PII detected: False


In [11]:
# task - 3
import joblib

model = joblib.load('best_model.pkl')
print("Model loaded successfully:", type(model))

Model loaded successfully: <class 'sklearn.pipeline.Pipeline'>


In [12]:
cut_order = {'Fair': 0, 'Good': 1, 'Very Good': 2, 'Premium': 3, 'Ideal': 4}
color_order = {'J': 0, 'I': 1, 'H': 2, 'G': 3, 'F': 4, 'E': 5, 'D': 6}
clarity_order = {'I1': 0, 'SI2': 1, 'SI1': 2, 'VS2': 3, 'VS1': 4, 'VVS2': 5, 'VVS1': 6, 'IF': 7}

def encode_record(features):
    encoded = features.copy()
    encoded['cut'] = cut_order[encoded['cut']]
    encoded['color'] = color_order[encoded['color']]
    encoded['clarity'] = clarity_order[encoded['clarity']]

    df_row = pd.DataFrame([encoded])
    df_row = df_row[['carat', 'cut', 'color', 'clarity', 'depth', 'table', 'x', 'y', 'z']]
    return df_row

In [13]:
diamond_1 = {
    'carat': 1.5,
    'cut': 'Ideal',
    'color': 'D',
    'clarity': 'VVS2',
    'depth': 61.5,
    'table': 57.0,
    'x': 7.3,
    'y': 7.3,
    'z': 4.5
}

encoded_1 = encode_record(diamond_1)
print(encoded_1)

pred_class = model.predict(encoded_1)[0]
pred_proba = model.predict_proba(encoded_1)[0][1]

print("Predicted class:", pred_class)
print("Predicted probability (expensive):", pred_proba)

   carat  cut  color  clarity  depth  table    x    y    z
0    1.5    4      6        5   61.5   57.0  7.3  7.3  4.5
Predicted class: 1
Predicted probability (expensive): 1.0


In [14]:
diamond_2 = {
    'carat': 0.3,
    'cut': 'Good',
    'color': 'I',
    'clarity': 'SI1',
    'depth': 62.0,
    'table': 58.0,
    'x': 4.3,
    'y': 4.3,
    'z': 2.7
}

diamond_3 = {
    'carat': 0.7,
    'cut': 'Very Good',
    'color': 'G',
    'clarity': 'VS1',
    'depth': 61.8,
    'table': 57.5,
    'x': 5.7,
    'y': 5.7,
    'z': 3.5
}

test_diamonds = [diamond_1, diamond_2, diamond_3]
predictions_data = []

for i, d in enumerate(test_diamonds, 1):
    encoded = encode_record(d)
    pred_class = model.predict(encoded)[0]
    pred_proba = model.predict_proba(encoded)[0][1]
    predictions_data.append({
        'diamond': d,
        'pred_class': pred_class,
        'pred_proba': pred_proba
    })
    print(f"Diamond {i}: class={pred_class}, probability={pred_proba:.4f}")

Diamond 1: class=1, probability=1.0000
Diamond 2: class=0, probability=0.0000
Diamond 3: class=1, probability=0.8961


In [15]:
system_prompt = """You are an expert diamond pricing analyst. You will be given a diamond's feature values, a machine learning model's predicted class (0 = not expensive, 1 = expensive, relative to the median price), and the model's predicted probability of the diamond being expensive.

Your task is to explain this prediction in plain language for a non-technical audience. Output ONLY valid JSON, with no additional text, markdown formatting, or explanation outside the JSON object.

The JSON object must have exactly these 5 fields:
{
  "prediction_label": "string - either 'Expensive' or 'Not Expensive'",
  "confidence_level": "string - one of 'low', 'medium', or 'high', based on how close the probability is to 0.5 (near 0.5 = low, near 0 or 1 = high)",
  "top_reason": "string - the single most important feature driving this prediction, explained in plain language",
  "second_reason": "string - the second most important feature driving this prediction, explained in plain language",
  "next_step": "string - a brief, practical recommendation for someone evaluating this diamond, based on the prediction"
}
"""

In [16]:
explanation_schema = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string"},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"}
    },
    "required": ["prediction_label", "confidence_level", "top_reason", "second_reason", "next_step"]
}

In [17]:
try:
    import jsonschema
    print("jsonschema available")
except ImportError:
    !pip install jsonschema
    import jsonschema
    print("jsonschema installed and imported")

jsonschema available


In [18]:
def build_user_prompt(features, pred_class, pred_proba):
    label = "Expensive" if pred_class == 1 else "Not Expensive"
    prompt = f"""Diamond features:
- Carat: {features['carat']}
- Cut: {features['cut']}
- Color: {features['color']}
- Clarity: {features['clarity']}
- Depth: {features['depth']}
- Table: {features['table']}
- Dimensions (x, y, z): {features['x']}mm, {features['y']}mm, {features['z']}mm

Model prediction: {label}
Predicted probability of being expensive: {pred_proba:.4f}

Explain this prediction as a JSON object with the 5 required fields."""
    return prompt

In [19]:
import json
from jsonschema import validate, ValidationError

def get_explanation(features, pred_class, pred_proba):
    user_prompt = build_user_prompt(features, pred_class, pred_proba)

    # PII guardrail check
    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return None

    response = call_llm(system_prompt, user_prompt, temperature=0.0)

    if response is None:
        print("LLM call failed.")
        return {k: None for k in explanation_schema['required']}

    response_clean = response.strip()

    try:
        parsed = json.loads(response_clean)
    except json.JSONDecodeError as e:
        print("JSON decode error:", e)
        return {k: None for k in explanation_schema['required']}

    try:
        validate(instance=parsed, schema=explanation_schema)
        print("Validation: PASS")
        return parsed
    except ValidationError as e:
        print("Validation error:", e.message)
        return {k: None for k in explanation_schema['required']}

In [20]:
results_table = []

for i, item in enumerate(predictions_data, 1):
    print(f"\n--- Diamond {i} ---")
    explanation = get_explanation(item['diamond'], item['pred_class'], item['pred_proba'])
    print("Explanation:", explanation)

    results_table.append({
        'diamond': item['diamond'],
        'pred_class': item['pred_class'],
        'pred_proba': item['pred_proba'],
        'explanation': explanation
    })


--- Diamond 1 ---
Validation: PASS
Explanation: {'prediction_label': 'Expensive', 'confidence_level': 'high', 'top_reason': 'The diamond is large at 1.5 carats, which is significantly above average size for diamonds.', 'second_reason': 'It also has a near-perfect color grade (D) and clarity (VVS2), indicating exceptional quality.', 'next_step': 'Given its high quality and size, you may want to proceed with a purchase or detailed evaluation, but verify the certification and market pricing first.'}

--- Diamond 2 ---
JSON decode error: Unterminated string starting at: line 6 column 16 (char 304)
Explanation: {'prediction_label': None, 'confidence_level': None, 'top_reason': None, 'second_reason': None, 'next_step': None}

--- Diamond 3 ---
JSON decode error: Expecting value: line 1 column 1 (char 0)
Explanation: {'prediction_label': None, 'confidence_level': None, 'top_reason': None, 'second_reason': None, 'next_step': None}


In [21]:
def clean_json_response(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.lower().startswith("json"):
            text = text[4:]
    return text.strip()

In [22]:
# Temperature A/B comparison
temp_comparison = []
for i, item in enumerate(predictions_data, 1):
    user_prompt = build_user_prompt(item['diamond'], item['pred_class'], item['pred_proba'])
    out_t0 = call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=800)
    out_t07 = call_llm(system_prompt, user_prompt, temperature=0.7, max_tokens=800)
    temp_comparison.append({'diamond': i, 'temp0': out_t0, 'temp0.7': out_t07})
    print(f"--- Diamond {i} ---")
    print("temp=0:  ", out_t0)
    print("temp=0.7:", out_t07)
    print()

--- Diamond 1 ---
temp=0:   None
temp=0.7: {
  "prediction_label": "Expensive",
  "confidence_level": "high",
  "top_reason": "The diamond has a high carat weight of 1.5, which is significantly above the average carat weight and contributes significantly to its value.",
  "second_reason": "The diamond has an 'Ideal' cut, which means it reflects light exceptionally well, increasing its brilliance and desirability.",
  "next_step": "Given the high prediction of being expensive and the high confidence level, it is recommended to compare the price of this diamond with other diamonds of similar quality and size to ensure a fair valuation."
}


--- Diamond 2 ---
temp=0:   {
  "prediction_label": "Not Expensive",
  "confidence_level": "high",
  "top_reason": "The diamond’s small size (0.3 carat) makes it less valuable.",
  "second_reason": "Its lower color grade (I) reduces its overall worth.",
  "next_step": "Since the price is low, you may want to check if the size and appearance meet your 